# 02 · Training-only feature selection and preprocessing audit

This notebook chooses the feature-selection **recipe**, not a globally precomputed feature matrix for later model evaluation. Selection is fitted on the training partition for inspection; notebooks 03–04 refit it inside every chronological CV fold through `FraudPipeline`.

GPU mode uses LightGBM feature selection by default. Set `FRAUD_NOTEBOOK_GPU=0` to use the original CPU consensus method. The same selection recipe is refitted inside each modeling fold.


## 1. Load prepared data and the feature recipe


In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Locate the repository from either the project root or notebook/.
current_directory = Path.cwd().resolve()
PROJECT_ROOT = None
for directory in [current_directory, *current_directory.parents]:
    if (directory / "src" / "fraud_detection").is_dir():
        PROJECT_ROOT = directory
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root. Expected src/fraud_detection."
    )

source_path = str(PROJECT_ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

SEED = 42
SMOKE = os.environ.get("FRAUD_NOTEBOOK_SMOKE", "0") == "1"
MAX_ROWS = int(os.environ.get("FRAUD_NOTEBOOK_MAX_ROWS", "0"))
if MAX_ROWS < 0:
    raise ValueError("FRAUD_NOTEBOOK_MAX_ROWS must be nonnegative.")

stage_setting = os.environ.get(
    "FRAUD_NOTEBOOK_STAGE_DIR", "data/processed/learning_stages"
)
STAGE_ROOT = Path(stage_setting).expanduser()
if not STAGE_ROOT.is_absolute():
    STAGE_ROOT = PROJECT_ROOT / STAGE_ROOT
STAGE_ROOT = STAGE_ROOT.resolve()
STAGE_ROOT.mkdir(parents=True, exist_ok=True)

print({
    "project_root": str(PROJECT_ROOT),
    "stage_root": str(STAGE_ROOT),
    "smoke": SMOKE,
    "max_rows": MAX_ROWS,
    "seed": SEED,
})

from fraud_detection.data_preparation import load_prepared_data
from fraud_detection.data import calculate_file_sha256
from fraud_detection.utils import frame_fingerprint

pointer_path = STAGE_ROOT / "prepared_latest.json"
if not pointer_path.is_file():
    raise FileNotFoundError(
        f"Missing {pointer_path}. Run notebook 00_data_preparation.ipynb first."
    )
pointer = json.loads(pointer_path.read_text(encoding="utf-8"))

if bool(pointer.get("smoke")) != SMOKE:
    raise ValueError(
        "Notebook mode differs from notebook 00. Use the same FRAUD_NOTEBOOK_SMOKE setting."
    )
if int(pointer.get("max_rows", 0)) != MAX_ROWS:
    raise ValueError(
        "FRAUD_NOTEBOOK_MAX_ROWS differs from notebook 00. Use the same row limit."
    )

prepared_folder = Path(pointer["prepared_folder"])
prepared_data = load_prepared_data(prepared_folder)
manifest_path = prepared_folder / "prepared_manifest.json"
actual_manifest_sha256 = calculate_file_sha256(manifest_path)
if actual_manifest_sha256 != pointer["prepared_manifest_sha256"]:
    raise ValueError("Prepared manifest changed since notebook 00.")

partitions = prepared_data["partitions"]
labeled_pool = prepared_data["labeled_pool"]
kaggle_test = prepared_data["kaggle_test"]
DATA_SCOPE = prepared_data["manifest"]["data_scope"]
if DATA_SCOPE != pointer["data_scope"]:
    raise ValueError("Prepared data scope differs from notebook 00 pointer.")

current_fingerprints = {
    name: frame_fingerprint(getattr(partitions, name))
    for name in ("train", "validation", "holdout")
}
if current_fingerprints != pointer["partition_fingerprints"]:
    raise ValueError("Prepared partition fingerprints differ from notebook 00.")

SOURCE_FILE_SHA256 = dict(pointer.get("source_file_sha256", {}))
JOIN_QUALITY_REPORT = prepared_data.get("quality_report", {})

print({
    "data_scope": DATA_SCOPE,
    "prepared_folder": str(prepared_folder),
    "train_rows": len(partitions.train),
    "validation_rows": len(partitions.validation),
    "holdout_rows": len(partitions.holdout),
    "kaggle_test_rows": len(kaggle_test),
    "integrity": "checksum + shape + partition fingerprints verified",
})

feature_recipe_path = STAGE_ROOT / "feature_recipe.json"
if not feature_recipe_path.is_file():
    raise FileNotFoundError("Run 01_feature_engineering.ipynb first.")
feature_recipe = json.loads(feature_recipe_path.read_text(encoding="utf-8"))
if feature_recipe["prepared_manifest_sha256"] != pointer["prepared_manifest_sha256"]:
    raise ValueError("Feature recipe belongs to a different prepared dataset.")
if feature_recipe["partition_fingerprints"] != pointer["partition_fingerprints"]:
    raise ValueError("Feature recipe partition lineage differs.")
display(pd.Series({k:v for k,v in feature_recipe.items() if k not in ("engineered_features", "raw_screen_report")}, name="feature_recipe"))


## 2. Refit the exploratory engineering stage on training


In [ ]:
from fraud_detection.features import FeatureEngineering, RawFeatureScreen

X_train = partitions.train.drop(columns="isFraud")
y_train = partitions.train.isFraud

screen = RawFeatureScreen(
    raw_screen_strategy=feature_recipe["raw_screen_strategy"],
    top_v_features=feature_recipe["top_v_features"],
    seed=SEED,
).fit(X_train, y_train)
screened_train = screen.transform(X_train)

# FeatureSelection requires numeric encoded inputs; use the same exploratory
# encoding that notebook 01 used for this learning-only inspection.
engineer = FeatureEngineering(
    encoding=feature_recipe.get("exploration_encoding", "legacy"),
    random_state=SEED,
    feature_groups=feature_recipe["feature_groups"],
    n_clusters=feature_recipe["n_clusters"],
).fit(screened_train, y_train)
engineered_train = engineer.transform(screened_train)
print("engineered training shape:", engineered_train.shape)


## 3. Fit the chosen selection method


In [ ]:
from fraud_detection.feature_selection import (
    DEFAULT_CONSENSUS_COMPONENTS,
    FeatureSelection,
    compare_selection_methods,
    feature_selection_stability,
)

GPU_ENABLED = os.environ.get("FRAUD_NOTEBOOK_GPU", "1") == "1"
SELECTION_DEVICE = "gpu" if GPU_ENABLED else "cpu"
SELECTION_METHOD = os.environ.get(
    "FRAUD_NOTEBOOK_SELECTION_METHOD", "lightgbm" if GPU_ENABLED else "consensus"
)
print({"gpu_enabled": GPU_ENABLED, "selection_method": SELECTION_METHOD, "selection_device": SELECTION_DEVICE})
MAX_FEATURES_RAW = int(os.environ.get("FRAUD_NOTEBOOK_MAX_FEATURES", "0"))
MAX_FEATURES = MAX_FEATURES_RAW or None
PRESERVE_FEATURES = tuple(
    x.strip() for x in os.environ.get("FRAUD_NOTEBOOK_PRESERVE_FEATURES", "").split(",")
    if x.strip()
)
COMPONENTS = tuple(
    x.strip() for x in os.environ.get(
        "FRAUD_NOTEBOOK_CONSENSUS_COMPONENTS",
        ",".join(DEFAULT_CONSENSUS_COMPONENTS),
    ).split(",") if x.strip()
)

selector = FeatureSelection(
    method=SELECTION_METHOD,
    device=SELECTION_DEVICE,
    max_features=MAX_FEATURES,
    random_state=SEED,
    preserve_features=PRESERVE_FEATURES,
    consensus_components=COMPONENTS,
).fit(engineered_train, y_train)
selected_train = selector.transform(engineered_train)

display(pd.Series(selector.selection_report_, name="selection_report"))
display(selector.importances_.head(30))
print("selected features:", len(selector.selected_features_))


## 4. Optional method comparison and chronological stability diagnostics


In [ ]:
RUN_DEEP_SELECTION = os.environ.get("FRAUD_NOTEBOOK_DEEP_SELECTION", "0") == "1"
if RUN_DEEP_SELECTION:
    methods = (
        "basic", "target_correlation", "random_forest", "lightgbm",
        "gradient_boosting", "permutation", "l1", "rfe", "shap", "consensus",
    )
    method_table, fitted_selectors = compare_selection_methods(
        engineered_train,
        y_train,
        methods=methods,
        device=SELECTION_DEVICE,
        max_features=MAX_FEATURES,
        random_state=SEED,
        consensus_components=COMPONENTS,
    )
    display(method_table)

    stability = feature_selection_stability(
        X_train,
        y_train,
        method=SELECTION_METHOD,
        selection_device=SELECTION_DEVICE,
        encoding=feature_recipe.get("exploration_encoding", "legacy"),
        max_features=MAX_FEATURES,
        random_state=SEED,
        feature_groups=feature_recipe["feature_groups"],
        n_clusters=feature_recipe["n_clusters"],
        raw_screen_strategy=feature_recipe["raw_screen_strategy"],
        top_v_features=feature_recipe["top_v_features"],
        preserve_features=PRESERVE_FEATURES,
        selection_consensus_components=COMPONENTS,
    )
    print("stability diagnostics computed")
else:
    print("Deep selection diagnostics skipped. Set FRAUD_NOTEBOOK_DEEP_SELECTION=1 to run them.")


## 5. Audit model-specific preprocessing without leaking later partitions


In [ ]:
from fraud_detection.preprocessing import Preprocessor

linear_preprocessor = Preprocessor(scale=True, scale_categorical=False).fit(selected_train)
tree_preprocessor = Preprocessor(scale=False, scale_categorical=False).fit(selected_train)

linear_train = linear_preprocessor.transform(selected_train)
tree_train = tree_preprocessor.transform(selected_train)
assert list(linear_train.columns) == selector.selected_features_
assert list(tree_train.columns) == selector.selected_features_
print({
    "training_medians_frozen": len(linear_preprocessor.medians_),
    "linear_scaled_columns": len(linear_preprocessor.scaled_columns_),
    "tree_scaled_columns": len(tree_preprocessor.scaled_columns_),
})


## 6. Save the complete raw-row model recipe


In [ ]:
model_recipe = {
    "schema_version": 1,
    "prepared_manifest_sha256": pointer["prepared_manifest_sha256"],
    "partition_fingerprints": pointer["partition_fingerprints"],
    "data_scope": DATA_SCOPE,
    "seed": SEED,
    "encoding": feature_recipe.get("encoding", "auto"),
    "feature_groups": feature_recipe["feature_groups"],
    "n_clusters": feature_recipe["n_clusters"],
    "raw_screen_strategy": feature_recipe["raw_screen_strategy"],
    "top_v_features": feature_recipe["top_v_features"],
    "selection_method": SELECTION_METHOD,
    "selection_device": SELECTION_DEVICE,
    "selection_max_features": MAX_FEATURES,
    "preserve_features": list(PRESERVE_FEATURES),
    "selection_consensus_components": list(COMPONENTS),
    "exploratory_selected_feature_count": len(selector.selected_features_),
    "exploratory_selected_features": list(selector.selected_features_),
}
model_recipe_path = STAGE_ROOT / "model_recipe.json"
model_recipe_path.write_text(json.dumps(model_recipe, indent=2, default=str), encoding="utf-8")
display(pd.Series({k:v for k,v in model_recipe.items() if k != "exploratory_selected_features"}, name="model_recipe"))
print("Next: run 03_modeling_baselines.ipynb")


## Why the recipe—not the matrix—is handed forward

The selected feature names above are descriptive evidence from one training fit. For honest model comparison, notebook 03 passes the **selection policy** into each complete raw-row pipeline so each chronological fold learns its own encoders, rankings, medians and scaling from that fold's training rows.
